纯数值推导

In [ ]:
import numpy as np
import pandas as pd

# Fixed parameters
Imax = 2.5  # Maximum expression level
L = 1.0  # Ligand concentration

# File paths
input_path = "E:/Desktop/mamm/estimate.csv"  # Input file path
output_path = "E:/Desktop/mamm/params_shared.csv"  # Output file path

# Read data
df = pd.read_csv(input_path)
# Split ID column into DBD and LBD information
df[['DBD', 'LBD']] = df['ID'].str.split('-', expand=True)


def calc_f(x, L=1.0):
    """
    Calculate activation proportion in basal state
    
    Parameters:
        x: Intermediate variable, sqrt(1 + 8*L*k1)
        L: Ligand concentration (default 1.0)
    
    Returns:
        Activation proportion in basal state: (L/2) * ((x-1)/(x+1))
    """
    return (L / 2.0) * ((x - 1.0) / (x + 1.0))


def solve_kd(I0, P0, k1):
    """
    Calculate dissociation constant kd from basal state data
    
    Parameters:
        I0: Basal expression level
        P0: Expression level in basal state
        k1: Cooperativity constant
    
    Returns:
        Dissociation constant kd
    """
    # Calculate basal activation fraction
    frac0 = (P0 - I0) / Imax
    
    # Calculate x2 in basal state
    x2 = np.sqrt(1 + 8 * L * k1)
    
    # Calculate basal state activation proportion
    f2 = calc_f(x2, L)
    
    # Derive kd from formula: kd = frac0 / (f2 * (1 - frac0))
    kd = frac0 / (f2 * (1 - frac0))
    
    return kd


def solve_M(I0, Pmax, kd, I):
    """
    Calculate parameter M from maximum activation data
    
    Parameters:
        I0: Basal expression level
        Pmax: Maximum expression level
        kd: Dissociation constant
        I: Inducer concentration
    
    Returns:
        Parameter M
    """
    # Calculate maximum activation fraction
    frac_max = (Pmax - I0) / Imax
    
    # Calculate square of x1
    x1_sq = 1 + 8 * L * (frac_max / (1 - frac_max) / kd / (L / 2))
    
    # Calculate x1
    x1 = np.sqrt(x1_sq)
    
    # Derive M: M = (x1^2 - 1) / (8 * L * I^2)
    M = (x1**2 - 1) / (8 * L * (I**2))
    
    return M


# Phase 1: Calculate k1 for each LBD
LBD_params = []
for lbd, sub in df.groupby('LBD'):
    # Take average of all data for current LBD
    I0 = sub['I0'].mean()
    P0 = sub['P0'].mean()
    
    # Calculate basal activation fraction
    frac0 = (P0 - I0) / Imax
    
    # Calculate k1 using formula: k1 = |((1 + 2*frac0)/(1 - 2*frac0))^2 - 1| / (8*L)
    k1 = abs(((1 + 2 * frac0) / (1 - 2 * frac0))**2 - 1) / (8 * L)
    
    # Save result (M temporarily set to NaN)
    LBD_params.append({'LBD': lbd, 'k1': k1, 'M': np.nan})

# Convert LBD parameters to DataFrame
LBD_df = pd.DataFrame(LBD_params)


# Phase 2: Calculate kd for each DBD
DBD_params = []
for dbd, sub in df.groupby('DBD'):
    # Collect kd values for all LBD combinations with current DBD
    kd_list = []
    
    # For each data row of current DBD
    for _, row in sub.iterrows():
        # Get corresponding k1 value from LBD_df
        k1 = LBD_df.loc[LBD_df['LBD'] == row['LBD'], 'k1'].values[0]
        
        # Calculate kd for current combination
        kd_list.append(solve_kd(row['I0'], row['P0'], k1))
    
    # Take average as kd for current DBD
    kd = np.mean(kd_list)
    
    # Save result
    DBD_params.append({'DBD': dbd, 'kd': kd})

# Convert DBD parameters to DataFrame
DBD_df = pd.DataFrame(DBD_params)


# Phase 3: Calculate M for each LBD
for i, row in LBD_df.iterrows():
    lbd = row['LBD']
    
    # Get all data for current LBD
    sub = df[df['LBD'] == lbd]
    
    # Collect all M values
    M_list = []
    
    # For each data row of current LBD
    for _, r in sub.iterrows():
        # Get corresponding kd value from DBD_df
        kd = DBD_df.loc[DBD_df['DBD'] == r['DBD'], 'kd'].values[0]
        
        # Calculate M for current combination
        M_list.append(solve_M(r['I0'], r['Pmax'], kd, r['I']))
    
    # Take average as M for current LBD
    LBD_df.loc[i, 'M'] = np.mean(M_list)


# Merge results with original data
merged = df.merge(DBD_df, on='DBD').merge(LBD_df, on='LBD')

# Save to CSV file
merged.to_csv(output_path, index=False)

# Output results
print("\n--- kd for each DBD ---")
print(DBD_df)
print("\n--- k1 and M for each LBD ---")
print(LBD_df)

✅ 参数结果已保存到：E:/Desktop/mamm/params_shared.csv

--- DBD 对应的 kd ---
       DBD        kd
0       CI  2.858510
1    CI434  1.145680
2   LexA87  0.767386
3  LexAs14  0.522634
4  LexAs17  1.465472
5  LexAs23  0.516736
6   LexAs5  2.760391
7     PurR  0.826089

--- LBD 对应的 k1 和 M ---
          LBD        k1             M
0        BjaR  0.041497  4.446421e-07
1     CinRori  0.097020  2.277001e-01
2        DHBR  0.041310  1.698463e-04
3          ER  0.008381  4.231792e-01
4   ER no nls  0.004034  1.370448e+00
5        LasR  0.030870  2.690705e-03
6   MR no nls  0.530136  1.525842e-01
7   PR no nls  0.057666  1.491990e-04
8        RpaR  0.013125  4.591401e-07
9        TraR  0.015293  1.477527e-04
10      acVHH  0.636593  4.267995e-01
